In [16]:
import pandas as pd
from pathlib import Path
import os
import sys

# força o Spark a usar o MESMO python (.venv) pro worker que roda o driver —
# sem isso, o worker sobe com o python de C:\spark\ (instalação separada,
# sem pyarrow instalado) e mapInPandas quebra com ModuleNotFoundError
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

# evita crash silencioso do worker Python no Windows quando numpy/scipy e
# pyarrow carregam runtimes OpenMP conflitantes no mesmo processo — precisa
# ser setado ANTES da SparkSession, pra propagar aos subprocessos worker
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *

import numpy as np
from scipy.spatial import ConvexHull, QhullError

import plotly.graph_objects as go

pd.set_option('display.max_columns', None)

In [17]:
# traceback "bonito" do IPython quebra (TokenError) ao formatar erros vindos
# de frames com fonte dinâmica (ex: lambdas de F.filter/F.transform do Spark)
# no Python 3.13 — Plain evita isso e mostra o erro real
%xmode Plain

Exception reporting mode: Plain


In [18]:
# Criação da sessão Spark local
spark = (
    SparkSession
    .builder
    .config("spark.driver.memory", "4g") 
    .config("spark.executor.memory", "4g") 
    .config("spark.python.worker.faulthandler.enabled", "true") # se algum worker crashar de novo, imprime o traceback nativo real
    .master("local[*]")
    .appName("feature_engineering")
    .getOrCreate()
)

## Funções

In [19]:
# Cada função de feature recebe o mesmo "context": um dict com os arrays
# achatados daquela linha (chaves = nomes das colunas temporárias criadas
# antes do mapInPandas). Retorna um único valor (float ou np.nan).


def _clean_xy(xs, ys):
    """Remove pares (x, y) com NaN — tracking ausente pra aquele jogador."""
    xs = np.asarray(xs, dtype=float)
    ys = np.asarray(ys, dtype=float)
    valid = ~(np.isnan(xs) | np.isnan(ys))
    return xs[valid], ys[valid]


def compute_surface_area(context):
    """Feature 1: área do casco convexo dos defensores de linha (m²)."""
    xs, ys = _clean_xy(context["defending_outfield_x"], context["defending_outfield_y"])
    if len(xs) < 3:
        return np.nan
    pts = np.array(sorted(set(zip(xs.tolist(), ys.tolist()))), dtype=float)
    if len(pts) < 3:
        return np.nan
    try:
        area = ConvexHull(pts).volume  # em 2D, .volume = área (.area seria o perímetro)
        return round(float(area), 2)
    except QhullError:
        return np.nan  # pontos colineares -> casco degenerado


def compute_stretch_index(context):
    """Feature 2: distância média dos defensores de linha até o centroide (m)."""
    xs, ys = _clean_xy(context["defending_outfield_x"], context["defending_outfield_y"])
    if len(xs) == 0:
        return np.nan
    cx, cy = xs.mean(), ys.mean()
    stretch_index = np.mean(np.sqrt((xs - cx) ** 2 + (ys - cy) ** 2))
    return round(float(stretch_index), 2)


def compute_team_length(context):
    """Feature 3: distância horizontal (eixo x) entre o defensor de linha mais atrás e o mais à frente (m)."""
    xs, _ = _clean_xy(context["defending_outfield_x"], context["defending_outfield_y"])
    if len(xs) == 0:
        return np.nan
    return round(float(xs.max() - xs.min()), 2)


def compute_height_goal_player(context):
    """
    Feature 4: distância horizontal (eixo x) entre o gol do time defendendo e
    o defensor de linha mais próximo desse gol — mede a altura da linha
    defensiva. Ataque sempre normalizado pra direita (target_engineering.ipynb),
    logo o gol do time defendendo fica sempre em x = +stadiumLength/2, e o
    defensor mais próximo dele é o de maior x.
    """
    xs, _ = _clean_xy(context["defending_outfield_x"], context["defending_outfield_y"])
    if len(xs) == 0 or pd.isna(context["stadiumLength"]):
        return np.nan
    goal_x = context["stadiumLength"] / 2
    return round(float(goal_x - xs.max()), 2)


def _count_within_radius(xs, ys, ball_x, ball_y, radius):
    xs, ys = _clean_xy(xs, ys)
    return int(((xs - ball_x) ** 2 + (ys - ball_y) ** 2 <= radius ** 2).sum())


def _numeric_superiority(context, radius):
    bx, by = context["ball_x"], context["ball_y"]
    if pd.isna(bx) or pd.isna(by):
        return np.nan
    defenders = _count_within_radius(context["defending_x"], context["defending_y"], bx, by, radius)
    attackers = _count_within_radius(context["attacking_x"], context["attacking_y"], bx, by, radius)
    return defenders - attackers


def compute_numeric_superiority_10m(context):
    """Feature 6: defensores menos atacantes num raio de 10m da bola."""
    return _numeric_superiority(context, radius=10)


def compute_numeric_superiority_20m(context):
    """Feature 7: defensores menos atacantes num raio de 20m da bola."""
    return _numeric_superiority(context, radius=20)


# nome da coluna (bate com FEATURE_COLUMNS) -> função que calcula ela
FEATURE_FUNCS = {
    "surface_area": compute_surface_area,
    "stretch_index": compute_stretch_index,
    "team_length": compute_team_length,
    "height_goal_player": compute_height_goal_player,
    "numeric_superiority_10m": compute_numeric_superiority_10m,
    "numeric_superiority_20m": compute_numeric_superiority_20m,
}

In [20]:
def compute_defensive_features(iterator):
    """
    Função a ser passada pro mapInPandas: roda por lote (batch) de cada
    partição via Arrow nos executors — nunca materializa o df inteiro no
    driver, ao contrário de toPandas(). Só orquestra: monta o context de
    cada linha e chama cada FEATURE_FUNCS — o cálculo em si vive nas funções
    compute_*, então isso não cresce conforme mais features são adicionadas.
    Depende de TEMP_COLS e FEATURE_COLUMNS (definidas mais adiante, depois
    que o df é montado) já existirem no escopo global quando for chamada.

    Parâmetros
    ----------
    iterator : Iterator[pandas.DataFrame]
        Lotes de linhas de uma partição, cada um já com as colunas
        temporárias achatadas (TEMP_COLS).

    Yields
    ------
    pandas.DataFrame
        Cada lote de entrada, sem as colunas temporárias, com uma coluna a
        mais por feature em FEATURE_COLUMNS.
    """
    for pdf in iterator:
        n = len(pdf)
        results = {name: np.full(n, np.nan) for name, _ in FEATURE_COLUMNS}

        rows = zip(
            pdf["defending_outfield_x"], pdf["defending_outfield_y"],
            pdf["defending_x"], pdf["defending_y"],
            pdf["attacking_x"], pdf["attacking_y"],
            pdf["ball_x"], pdf["ball_y"],
            pdf["stadiumLength"],
        )
        for i, (out_x, out_y, def_x, def_y, atk_x, atk_y, bx, by, stadium_length) in enumerate(rows):
            context = {
                "defending_outfield_x": out_x, "defending_outfield_y": out_y,
                "defending_x": def_x, "defending_y": def_y,
                "attacking_x": atk_x, "attacking_y": atk_y,
                "ball_x": bx, "ball_y": by,
                "stadiumLength": stadium_length,
            }
            for name, func in FEATURE_FUNCS.items():
                results[name][i] = func(context)

        pdf = pdf.drop(columns=TEMP_COLS)
        for name, values in results.items():
            pdf[name] = values

        yield pdf

In [22]:
# caminho pra pasta com dados
data_folder_path = Path().resolve().parent.parent / "data"

In [23]:
# base de ameaça criada
threat_dataset_path = str(data_folder_path / "threat_dataset")

df = spark.read.parquet(threat_dataset_path)

In [24]:
# Só o necessário pro cálculo das features
df = df.select('competitionId', 'season', 'gameId', 'eventId', 'attackingPlayersNorm', 'defendingPlayersNorm', 'ballsNorm', 'stadiumLength')

df.show(2)

+-------------+---------+------+--------------------+--------------------+--------------------+--------------------+-------------+
|competitionId|   season|gameId|             eventId|attackingPlayersNorm|defendingPlayersNorm|           ballsNorm|stadiumLength|
+-------------+---------+------+--------------------+--------------------+--------------------+--------------------+-------------+
|            1|2022-2023|  4436|432ce6adb8330a937...|[{32.936, 29.208,...|[{13.396, 24.316,...|[{20.81, 29.59, 4...|        101.0|
|            1|2022-2023|  4436|d1d90634dee1eb108...|[{33.128, 29.179,...|[{13.563, 24.07, ...|[{20.81, 29.59, 4...|        101.0|
+-------------+---------+------+--------------------+--------------------+--------------------+--------------------+-------------+
only showing top 2 rows


In [25]:
df.count()

448193

In [26]:
# Achata array<struct> em array<float> de x/y antes do mapInPandas — reduz o
# payload serializado via Arrow (dropa player.id/name/visibility/confidence/
# position, que não entram nos cálculos). GK já sai filtrado aqui pro
# defendingOutfield, usado nas Features 1/2 (casco/stretch); defending_x/y
# e attacking_x/y (com GK) e ball_x/y alimentam as Features 6/7.
# position agora é um struct (type/typeDescription/groupType) — usa .type
# pra comparar com "GK", igual ao "GK" que já vinha da sigla original.
defending_outfield = F.filter("defendingPlayersNorm", lambda p: p["position"]["type"] != "GK")

df = df.withColumns({
    "defending_outfield_x": F.transform(defending_outfield, lambda p: p["x"]),
    "defending_outfield_y": F.transform(defending_outfield, lambda p: p["y"]),
    "defending_x": F.transform("defendingPlayersNorm", lambda p: p["x"]),
    "defending_y": F.transform("defendingPlayersNorm", lambda p: p["y"]),
    "attacking_x": F.transform("attackingPlayersNorm", lambda p: p["x"]),
    "attacking_y": F.transform("attackingPlayersNorm", lambda p: p["y"]),
    "ball_x": F.get("ballsNorm", 0)["x"],
    "ball_y": F.get("ballsNorm", 0)["y"],
}).drop("attackingPlayersNorm", "defendingPlayersNorm", "ballsNorm")

df.printSchema()

root
 |-- competitionId: long (nullable = true)
 |-- season: string (nullable = true)
 |-- gameId: long (nullable = true)
 |-- eventId: string (nullable = true)
 |-- stadiumLength: float (nullable = true)
 |-- defending_outfield_x: array (nullable = true)
 |    |-- element: float (containsNull = true)
 |-- defending_outfield_y: array (nullable = true)
 |    |-- element: float (containsNull = true)
 |-- defending_x: array (nullable = true)
 |    |-- element: float (containsNull = true)
 |-- defending_y: array (nullable = true)
 |    |-- element: float (containsNull = true)
 |-- attacking_x: array (nullable = true)
 |    |-- element: float (containsNull = true)
 |-- attacking_y: array (nullable = true)
 |    |-- element: float (containsNull = true)
 |-- ball_x: float (nullable = true)
 |-- ball_y: float (nullable = true)



In [27]:
# Colunas temporárias (arrays achatados + stadiumLength) criadas só pra
# alimentar o cálculo das features — descartadas do schema de saída do
# mapInPandas, que fica só com eventId + as features
TEMP_COLS = [
    "defending_outfield_x", "defending_outfield_y",
    "defending_x", "defending_y",
    "attacking_x", "attacking_y",
    "ball_x", "ball_y",
    "stadiumLength",
]

# Uma linha por feature: nome da coluna de saída + tipo Spark. Pra adicionar
# uma feature nova: escreve a função dela na célula de baixo, adiciona uma
# linha aqui e uma entrada em FEATURE_FUNCS — o resto (schema, orquestração)
# não muda.
FEATURE_COLUMNS = [
    ("surface_area", DoubleType()),            # Feature 1 — convex hull
    ("stretch_index", DoubleType()),           # Feature 2 — stretch index
    ("team_length", DoubleType()),             # Feature 3 — comprimento do time
    ("height_goal_player", DoubleType()),      # Feature 4 — altura da defesa (gol -> defensor mais recuado)
    ("numeric_superiority_10m", DoubleType()), # Feature 6 — superioridade 10m
    ("numeric_superiority_20m", DoubleType()), # Feature 7 — superioridade 20m
]

# Schema de saída = eventId (única coluna que sobra de df.schema depois de
# tirar TEMP_COLS) + as features
FEATURE_SCHEMA = StructType(
    [f for f in df.schema.fields if f.name not in TEMP_COLS]
    + [StructField(name, dtype) for name, dtype in FEATURE_COLUMNS]
)

In [28]:
df_features = df.mapInPandas(compute_defensive_features, schema=FEATURE_SCHEMA)

In [29]:
# output já é só eventId + as features (schema enxuto)
df_features.show(10, truncate=False)

+-------------+---------+------+--------------------------------+------------+-------------+-----------+------------------+-----------------------+-----------------------+
|competitionId|season   |gameId|eventId                         |surface_area|stretch_index|team_length|height_goal_player|numeric_superiority_10m|numeric_superiority_20m|
+-------------+---------+------+--------------------------------+------------+-------------+-----------+------------------+-----------------------+-----------------------+
|1            |2022-2023|4436  |432ce6adb8330a937eb01b2994bba259|341.67      |9.39         |24.58      |12.52             |0.0                    |-1.0                   |
|1            |2022-2023|4436  |d1d90634dee1eb108becfd44a33f98d2|342.45      |9.4          |24.6       |12.33             |0.0                    |-1.0                   |
|1            |2022-2023|4436  |524549c5ec7bb98207c6a4b001ad5790|376.04      |9.54         |23.31      |15.69             |0.0              

In [30]:
output_path = str(Path().resolve().parent.parent / "data" / "features_dataset")
df_features.write.mode("overwrite").option("header", True).csv(output_path)